# 🧠 Aula 02c: Comparação de Modelos Regressores no Scikit-Learn

Nesta aula prática, faremos um embate de diferentes algoritmos de regressão disponíveis na biblioteca **Scikit-Learn**. O objetivo é analisar o comportamento de cada um em relação ao erro de treino e teste, e entender conceitos vitais como:
1. **Modelos Lineares:** Regressão linear analítica e baseada em Gradiente Descendente (SGD).
2. **Suporte a Vetores (SVR):** Uma abordagem geométrica robusta.
3. **Baseados em Vizinhança (KNN):** Regressão não-paramétrica por média local.
4. **Árvores de Decisão:** O perigo do **Overfitting** (sobreajuste) extremo.
5. **Ensembles (Random Forest):** A união de múltiplas árvores para melhor generalização.

<a href="https://colab.research.google.com/github/fboldt/aulasml/blob/master/aula02c%20-%20compara%C3%A7%C3%B5es%20de%20regressores.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Carrega o conjunto de dados de diabetes do Scikit-Learn
from sklearn.datasets import load_diabetes
data = load_diabetes()
X, y = data.data, data.target

In [ ]:
# Divide o dataset em treino (80%) e teste (20%) com seed fixada em 42
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Importa a métrica oficial de Erro Quadrático Médio (MSE)
from sklearn.metrics import mean_squared_error

In [ ]:
from sklearn.base import BaseEstimator, RegressorMixin
import numpy as np

def include_bias(X):
  return np.hstack((np.ones((X.shape[0], 1)), X))

class LinearRegressor(BaseEstimator, RegressorMixin):
  def fit(self, X, y):
    X = include_bias(X)
    self.w_ = np.linalg.pinv(X) @ y
    return self

  def predict(self, X):
    X = include_bias(X)
    y_pred = X @ self.w_
    return y_pred.reshape(X.shape[0], )

regressor = LinearRegressor()
regressor.fit(X_train, y_train)
y_pred = regressor.predict(X_train)
print(regressor.w_)
print(mean_squared_error(y_train, y_pred))

[ 151.34560454   37.90402135 -241.96436231  542.42875852  347.70384391
 -931.48884588  518.06227698  163.41998299  275.31790158  736.1988589
   48.67065743]
2868.5497028355776


#### 🔍 O que este bloco faz?
Ele avalia o desempenho do nosso modelo personalizado no conjunto de **Teste** (dados que o modelo nunca viu durante o treinamento).

#### 🎯 Qual a intenção pedagógica?
Medir a capacidade de generalização do modelo e verificar se há uma discrepância muito grande entre o erro de treino e o erro de teste.

In [ ]:
# Faz previsões para o conjunto de teste e calcula o MSE correspondente
y_pred = regressor.predict(X_test)
print("MSE no Teste (Custom):", mean_squared_error(y_test, y_pred))

2900.1936284934764


### 🏁 2. Regressão Linear Oficial do Scikit-Learn

#### 🔍 O que este bloco faz?
Ele treina a classe oficial `LinearRegression` do Scikit-Learn e calcula o MSE correspondente para os conjuntos de treino e teste.

#### 🎯 Qual a intenção pedagógica?
Confirmar que os resultados do Scikit-Learn são idênticos aos obtidos pela nossa classe personalizada baseada em Pseudoinversa. Isso valida a matemática por trás da nossa implementação!

In [ ]:
from sklearn.linear_model import LinearRegression

# Instancia e treina o modelo de regressão linear padrão do Scikit-Learn
regressor = LinearRegression()
regressor.fit(X_train, y_train)
y_pred = regressor.predict(X_train)

# Exibe e compara os erros obtidos nos dois conjuntos
print("MSE training:\t", mean_squared_error(y_train, y_pred))
print("MSE test:\t", mean_squared_error(y_test, regressor.predict(X_test)))

MSE training:	 2868.549702835577
MSE test:	 2900.193628493482


### 🛡️ 3. Regressão por Vetores de Suporte (LinearSVR)

#### 🔍 O que este bloco faz?
Ele treina um regressor baseado em Support Vector Machines (SVM) com um kernel linear (`LinearSVR`) e mede o erro resultante.

#### 🎯 Qual a intenção pedagógica?
Apresentar uma abordagem geométrica de regressão. Diferente da regressão de mínimos quadrados (que tenta minimizar a soma dos erros quadrados de todos os pontos), o SVR tenta conter o maior número de pontos possível dentro de uma margem ajustável ($\epsilon$). Pontos fora dessa margem são punidos. Observe o erro consideravelmente maior devido à falta de normalização ou ajuste de hiperparâmetros!

In [ ]:
from sklearn.svm import LinearSVR

# Instancia e treina o estimador de Support Vector Regression (SVR) com kernel linear
regressor = LinearSVR()
regressor.fit(X_train, y_train)
y_pred = regressor.predict(X_train)

# Avaliação do MSE no treino e teste
print("MSE training:\t", mean_squared_error(y_train, y_pred))
print("MSE test:\t", mean_squared_error(y_test, regressor.predict(X_test)))

MSE training:	 8224.55551370706
MSE test:	 6775.878001000562


### ⚡ 4. Regressão Linear com Gradiente Descendente Estocástico (SGDRegressor)

#### 🔍 O que este bloco faz?
Ele treina o modelo de regressão linear otimizado via Gradiente Descendente Estocástico (`SGDRegressor`) fornecido pelo Scikit-Learn.

#### 🎯 Qual a intenção pedagógica?
Apresentar a versão iterativa e escalável da Regressão Linear. O SGD atualiza os pesos usando uma amostra aleatória de cada vez, tornando-o ideal para datasets gigantescos onde a Equação Normal (que inverte matrizes enormes) seria computacionalmente inviável.

In [ ]:
from sklearn.linear_model import SGDRegressor

# Instancia o estimador iterativo SGD com limite máximo de 10.000 iterações
regressor = SGDRegressor(max_iter=10000)
regressor.fit(X_train, y_train)
y_pred = regressor.predict(X_train)

# Avaliação do MSE no treino e teste
print("MSE training:\t", mean_squared_error(y_train, y_pred))
print("MSE test:\t", mean_squared_error(y_test, regressor.predict(X_test)))

MSE training:	 2950.6380118430347
MSE test:	 2863.3497715638714


### 👥 5. Regressão Baseada em Vizinhança (K-Neighbors Regressor)

#### 🔍 O que este bloco faz?
Ele treina o classificador de vizinhos mais próximos (`KNeighborsRegressor`) configurado para buscar os 5 vizinhos mais próximos ($k=5$).

#### 🎯 Qual a intenção pedagógica?
Ilustrar um método de regressão **não-paramétrico**. Em vez de tentar ajustar uma reta matemática global (como a regressão linear), o KNN prevê o valor de um ponto calculando a média dos alvos dos $K$ pontos mais próximos na vizinhança.

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

# Instancia e treina o regressor KNN com vizinhança de 5 pontos
regressor = KNeighborsRegressor(n_neighbors=5)
regressor.fit(X_train, y_train)
y_pred = regressor.predict(X_train)

# Avaliação do MSE no treino e teste
print("MSE training:\t", mean_squared_error(y_train, y_pred))
print("MSE test:\t", mean_squared_error(y_test, regressor.predict(X_test)))

MSE training:	 2528.5918413597733
MSE test:	 3019.075505617978


### 🌳 6. Regressão com Árvores de Decisão (DecisionTreeRegressor)

#### 🔍 O que este bloco faz?
Ele treina uma árvore de decisão para regressão (`DecisionTreeRegressor`) sem limites de profundidade e calcula os erros correspondentes.

#### 🎯 Qual a intenção pedagógica?
Apresentar o fenômeno clássico do **Overfitting** (sobreajuste):
- O **MSE de treino é exatamente `0.0`**! A árvore memorizou perfeitamente todos os dados de treino criando ramificações ultraespecíficas.
- O **MSE de teste é muito alto (`~4872.20`)**, pior do que qualquer outro modelo que vimos!
Isso ilustra perfeitamente que memorizar não é o mesmo que aprender. O modelo falhou drasticamente em generalizar para dados novos.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

# Instancia e treina uma árvore de decisão de regressão sem limite de profundidade
regressor = DecisionTreeRegressor()
regressor.fit(X_train, y_train)
y_pred = regressor.predict(X_train)

# Avaliação do MSE no treino e teste
print("MSE training:\t", mean_squared_error(y_train, y_pred))
print("MSE test:\t", mean_squared_error(y_test, regressor.predict(X_test)))

MSE training:	 0.0
MSE test:	 4872.202247191011


### 🌲🌲 7. Regressão com Florestas Aleatórias (Random Forest Regressor)

#### 🔍 O que este bloco faz?
Ele treina uma floresta aleatória de regressão (`RandomForestRegressor`) com profundidade máxima das árvores restrita a 3 (`max_depth=3`).

#### 🎯 Qual a intenção pedagógica?
Apresentar o conceito de **Ensemble (Comitê de Modelos)**. Em vez de confiar em uma única árvore de decisão (que vimos sofrer de overfitting extremo), a Floresta Aleatória combina as predições de múltiplas árvores independentes (média de votos). Ao limitar a profundidade (`max_depth=3`), evitamos que as árvores memorizem ruídos. O resultado é um modelo muito mais estável, com erro de treino e teste bem mais equilibrados!

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Instancia e treina uma Floresta Aleatória com árvores podadas em profundidade 3
regressor = RandomForestRegressor(max_depth=3)
regressor.fit(X_train, y_train)
y_pred = regressor.predict(X_train)

# Avaliação do MSE no treino e teste
print("MSE training:\t", mean_squared_error(y_train, y_pred))
print("MSE test:\t", mean_squared_error(y_test, regressor.predict(X_test)))

MSE training:	 2530.820187105715
MSE test:	 2785.977900513713
